# Chapter 1 — Loading and Visualising Drone Data

This Jupyter notebook loads a 10-band **MicaSense RedEdge-P Dual** image, summarises its
spectral bands, creates true- and false-colour composites, and places
the drone image on an interactive basemap.

**Before starting:** copy the 10-band GeoTIFF to
`data/COG.tif`. Run the installation cell once, restart the kernel
if Jupyter asks, and then choose **Run the code**.


In [ ]:
# Run once in a new environment, then restart the kernel if prompted.
# Install the workshop dependencies from requirements.txt. Ensure it includes:
# Install dask[distributed] and rioxarray for chunked reads and the dashboard.
# Ensure requirements.txt contains these packages, or run once:

# %pip install "dask[distributed]" rioxarray
# %pip install -r requirements.txt


## 1. Load libraries and shared workshop functions


In [ ]:
from pathlib import Path
import sys

import dask
import dask.array as dask_array
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
import rioxarray as rxr
from IPython.display import display, Markdown, clear_output

pio.renderers.default = "plotly_mimetype+notebook_connected"

# Start a local Dask Client so workers and task activity are visible in the
# notebook. Threads avoid spawning extra Python processes and keep the worker
# setup reliable in Windows-based Jupyter environments.
from dask.distributed import Client, LocalCluster

DASK_WORKERS = 4
cluster = LocalCluster(
    n_workers=DASK_WORKERS,
    threads_per_worker=2,
    processes=False,
    dashboard_address=":0",
)
client = Client(cluster, set_as_default=True)
print(f"Dask scheduler: {client.scheduler.address}")
print(f"Dask workers: {len(client.scheduler_info()['workers'])}")
print(f"Dask dashboard: {client.dashboard_link}")
client

# The notebooks and workshop_utils.py should remain in the same folder.
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "workshop_utils.py").exists():
    raise FileNotFoundError(
        "Open Jupyter in the converted workshop folder so workshop_utils.py "
        "and requirements.txt are beside this notebook."
    )
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from plotly.subplots import make_subplots
from workshop_utils import (
    BAND_INFO, pstretch, array_to_png_b64, to_uint8,
    interactive_raster_explorer, plot_raster_distribution,
    projected_plot_context, make_projected_image, apply_projected_axes,
)

# Set the path to the GeoTIFF stored on your computer.
# Expected workshop layout: notebook_folder/data/your_raster.tif
RASTER_FILENAME = "NTABRT0007-20240828-ms-cog-subset.tif"
RASTER_PATH = NOTEBOOK_DIR / "data" / RASTER_FILENAME


# _HERE = Path(__file__).resolve().parent
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"

# Also allow the TIFF to sit beside the notebook.
if not RASTER_PATH.exists():
    local_file = NOTEBOOK_DIR / RASTER_FILENAME
    if local_file.exists():
        RASTER_PATH = local_file

if not RASTER_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find the raster at {RASTER_PATH}. "
        "Edit RASTER_FILENAME / RASTER_PATH above to match the location "
        "of your local GeoTIFF."
    )

print(f"Local raster: {RASTER_PATH.resolve()}")
print(f"Dask scheduler: {client.scheduler.address}")
print(f"Dask workers: {len(client.scheduler_info()['workers'])}")


## 2. Load the raster

Each spectral band is loaded as a variable in an `xarray.Dataset`.
Background pixels stored as zero are converted to `NaN`.


In [ ]:
# Open the GeoTIFF as lazy Dask chunks. Only chunks needed by a later
# calculation are read into memory. Keep one band per chunk to bound working RAM.
RASTER_CHUNKS = {"band": 1, "x": 512, "y": 512}
raster = rxr.open_rasterio(
    RASTER_PATH,
    masked=True,
    chunks=RASTER_CHUNKS,
    cache=False,
    lock=True,
).astype("float32")

expected_band_names = [band["var"] for band in BAND_INFO]
if raster.sizes.get("band") != len(expected_band_names):
    raster.close()
    raise ValueError(
        f"Expected {len(expected_band_names)} bands, found "
        f"{raster.sizes.get('band')} in {RASTER_PATH.name}."
    )

# The supplied image uses zero as background/NoData. Keep values lazy while
# masking those pixels and converting the band dimension to named variables.
raster = raster.assign_coords(band=expected_band_names)
raster = raster.where(np.isfinite(raster) & (raster > 0))
ds = raster.to_dataset(dim="band")
ds.attrs.update({
    "filename": RASTER_PATH.name,
    "source_path": str(RASTER_PATH.resolve()),
    "instrument": "MicaSense RedEdge-P Dual",
    "image_width_px": int(raster.sizes["x"]),
    "image_height_px": int(raster.sizes["y"]),
    "pixel_size_m": float(abs(raster.rio.resolution()[0])),
    "crs": str(raster.rio.crs),
    "transform": tuple(raster.rio.transform()),
    "bounds": tuple(float(v) for v in raster.rio.bounds()),
    "dask_chunks": str(RASTER_CHUNKS),
})
for band in BAND_INFO:
    ds[band["var"]].attrs.update({
        "long_name": band["name"],
        "centre_wavelength_nm": band["nm"],
        "fwhm_nm": band["fwhm"],
    })

ds


In [ ]:
# Check what is now in the dataset
print(list(ds.data_vars))

In [ ]:
from pprint import pprint

print("Dataset summary")
print(ds)

print("\nGlobal attributes")
pprint(dict(ds.attrs), sort_dicts=False)

print("\nDataset encoding")
pprint(dict(ds.encoding), sort_dicts=False)

print("\nCoordinates")
for name, coord in ds.coords.items():
    print(f"\n{name}: dims={coord.dims}, shape={coord.shape}, dtype={coord.dtype}")
    print("attrs:")
    pprint(dict(coord.attrs), sort_dicts=False)
    print("encoding:")
    pprint(dict(coord.encoding), sort_dicts=False)

print("\nData variables")
for name, var in ds.data_vars.items():
    print(f"\n{name}: dims={var.dims}, shape={var.shape}, dtype={var.dtype}")
    print("attrs:")
    pprint(dict(var.attrs), sort_dicts=False)
    print("encoding:")
    pprint(dict(var.encoding), sort_dicts=False)

# Spatial metadata, if rioxarray is available
try:
    print("\nCRS:", ds.rio.crs)
    print("Spatial dimensions:", ds.rio.x_dim, ds.rio.y_dim)
    print("NoData value:", ds.rio.nodata)
except Exception as e:
    print("\nCould not read rioxarray metadata:", e)

In [ ]:
# Build the same per-band invalid-pixel mask without materializing the stack.
# It remains lazy; cell 9 displays only the Dask array summary.
stacked = ds.to_array(dim="band")
invalid = (
    stacked.isnull()
    | ~np.isfinite(stacked)
    | (stacked == -32767)
    | (stacked < 0)
    | (stacked > 1)
)


In [ ]:
invalid

In [ ]:
print("Dataset attributes:", ds.attrs)
# print("Dataset encoding:", ds.encoding)

# for name, da in ds.data_vars.items():
#     print(f"\n{name}")
#     print("Attributes:", da.attrs)
#     print("Encoding:", da.encoding)

In [ ]:
# import rasterio

# with rasterio.open(DATA_PATH) as src:
#     print("NoData / ignore value:", src.nodata)
#     print("Dataset tags:", src.tags())

#     for band in range(1, src.count + 1):
#         print(f"Band {band} tags:", src.tags(band))

In [ ]:
# The raster is already opened above as a lazy, chunked rioxarray object.
# Inspect its storage layout without reading the full image into memory.
print("Raster chunks:", raster.data.chunksize)
print("Raster shape (band, y, x):", raster.shape)
print("Raster source:", ds.attrs["source_path"])


In [ ]:
# # Valid reflectance values are finite and greater than 0, up to and including 1.
# valid_mask = np.isfinite(ds) & (ds > 0) & (ds <= 1)

# # Mask invalid values in every band. Assign back to ds for later cells.
# ds = ds.where(valid_mask)
# ds

In [ ]:
invalid_mask = (
    ~np.isfinite(ds)
    | (ds == 0)
    | (ds < 0)
    | (ds > 1)
)

ds = ds.where(~invalid_mask)
ds

## 3. Band specification and pixel statistics


In [ ]:
# Reduce one band at a time. Dask reads bounded chunks and returns only scalar
# statistics, instead of creating a full-scene NumPy copy for each band.
rows = []
scene_means = []
for band in BAND_INFO:
    arr = ds[band["var"]].data
    finite = dask_array.isfinite(arr)
    valid_count_task = finite.sum(dtype="int64")
    min_task = dask_array.nanmin(arr)
    max_task = dask_array.nanmax(arr)
    mean_task = dask_array.nanmean(arr)
    valid_count, band_min, band_max, band_mean = dask.compute(
        valid_count_task, min_task, max_task, mean_task
    )
    scene_means.append(float(band_mean) if valid_count else np.nan)
    rows.append({
        "Band": band["var"], "Name": band["name"],
        "Centre (nm)": band["nm"], "FWHM (nm)": band["fwhm"],
        "Min": round(float(band_min), 4) if valid_count else np.nan,
        "Max": round(float(band_max), 4) if valid_count else np.nan,
        "Mean": round(float(band_mean), 4) if valid_count else np.nan,
        "Valid px": int(valid_count),
    })
band_stats = pd.DataFrame(rows)
display(band_stats)


## 4. Sensor spectral response

The curves approximate each band as a Gaussian centred on its nominal
wavelength. The red-edge bands lie where vegetation reflectance changes
rapidly, making them useful for plant-condition monitoring.


In [ ]:
def hex_to_rgba(hex_colour, alpha=0.18):
    """Convert '#RRGGBB' to a Plotly-compatible rgba() colour."""
    hex_colour = hex_colour.lstrip("#")
    red, green, blue = (
        int(hex_colour[i:i + 2], 16)
        for i in (0, 2, 4)
    )
    return f"rgba({red}, {green}, {blue}, {alpha})"


wl = np.linspace(420, 900, 1200)

fig_spec = go.Figure()

for band in BAND_INFO:
    sigma = band["fwhm"] / 2.355

    response = np.exp(
        -0.5 * ((wl - band["nm"]) / sigma) ** 2
    )

    fig_spec.add_trace(
        go.Scatter(
            x=wl,
            y=response,
            fill="tozeroy",
            fillcolor=hex_to_rgba(band["colour"], alpha=0.18),
            line=dict(
                color=band["colour"],
                width=2,
            ),
            name=f"{band['name']} ({band['nm']} nm)",
            hovertemplate=(
                f"<b>{band['name']}</b>"
                f"<br>Centre: {band['nm']} nm"
                f"<br>FWHM: {band['fwhm']} nm"
                "<br>Response: %{y:.3f}"
                "<extra></extra>"
            ),
        )
    )

fig_spec.update_layout(
    title="MicaSense RedEdge-P Dual — Spectral Band Positions",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Normalised Spectral Response",
    yaxis_range=[0, 1.2],
    height=420,
    legend=dict(
        font=dict(size=10),
        orientation="h",
        y=-0.25,
    ),
)

fig_spec.show()

## 5. True-colour and false-colour composites


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def norm(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-6)

rgb = np.dstack([
    norm(ds["red668"].values),
    norm(ds["green560"].values),
    norm(ds["blue"].values)
])

plt.imshow(rgb)
plt.axis("off")
plt.title("RGB (stretched)")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def percentile_stretch(band, low=2, high=98):
    """Stretch finite pixels between the chosen percentiles to [0, 1]."""
    values = np.asarray(band, dtype=np.float32)
    valid = np.isfinite(values)

    if not valid.any():
        return np.zeros(values.shape, dtype=np.float32)

    vmin, vmax = np.percentile(values[valid], [low, high])
    stretched = np.zeros(values.shape, dtype=np.float32)

    if vmax > vmin:
        stretched[valid] = np.clip(
            (values[valid] - vmin) / (vmax - vmin), 0, 1
        )

    return stretched

rgb = np.dstack([
    percentile_stretch(ds["red668"].values),   # Red
    percentile_stretch(ds["green560"].values), # Green
    percentile_stretch(ds["blue"].values),     # Blue
])

plt.imshow(rgb)
plt.axis("off")
plt.title("RGB (2nd–98th percentile stretch)")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def percentile_stretch(band, low=2, high=98):
    values = np.asarray(band, dtype=np.float32)
    valid = np.isfinite(values)

    if not valid.any():
        return np.zeros(values.shape, dtype=np.float32)

    vmin, vmax = np.percentile(values[valid], [low, high])
    stretched = np.zeros(values.shape, dtype=np.float32)

    if vmax > vmin:
        stretched[valid] = np.clip(
            (values[valid] - vmin) / (vmax - vmin), 0, 1
        )

    return stretched

rgb = np.dstack([
    percentile_stretch(ds["red668"].values),    # Red
    percentile_stretch(ds["green560"].values),  # Green
    percentile_stretch(ds["blue"].values),      # Blue
])

# Get pixel-centre coordinates and raster bounds.
x = np.asarray(ds["x"].values)
y = np.asarray(ds["y"].values)
dx = abs(float(np.median(np.diff(x))))
dy = abs(float(np.median(np.diff(y))))

left, right = x.min() - dx / 2, x.max() + dx / 2
bottom, top = y.min() - dy / 2, y.max() + dy / 2

fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(rgb, extent=[left, right, bottom, top], origin="upper")
ax.set_xlabel("Easting")
ax.set_ylabel("Northing")
ax.set_title("RGB (2nd–98th percentile stretch)")
plt.tight_layout()
plt.show()

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.ticker import FuncFormatter
# from pyproj import Transformer

# def percentile_stretch(band, low=2, high=98):
#     values = np.asarray(band, dtype=np.float32)
#     valid = np.isfinite(values)

#     if not valid.any():
#         return np.zeros(values.shape, dtype=np.float32)

#     vmin, vmax = np.percentile(values[valid], [low, high])
#     stretched = np.zeros(values.shape, dtype=np.float32)

#     if vmax > vmin:
#         stretched[valid] = np.clip(
#             (values[valid] - vmin) / (vmax - vmin), 0, 1
#         )

#     return stretched

# rgb = np.dstack([
#     percentile_stretch(ds["red668"].values),    # Red
#     percentile_stretch(ds["green560"].values),  # Green
#     percentile_stretch(ds["blue"].values),      # Blue
# ])

# # Get pixel-centre coordinates and raster bounds.
# x = np.asarray(ds["x"].values)
# y = np.asarray(ds["y"].values)
# dx = abs(float(np.median(np.diff(x))))
# dy = abs(float(np.median(np.diff(y))))

# left, right = x.min() - dx / 2, x.max() + dx / 2
# bottom, top = y.min() - dy / 2, y.max() + dy / 2
# centre_x = float((left + right) / 2)
# centre_y = float((bottom + top) / 2)

# # Convert projected coordinates to longitude and latitude.
# transformer = Transformer.from_crs(
#     ds.attrs["crs"], "EPSG:4326", always_xy=True
# )

# fig, ax = plt.subplots(figsize=(9, 8))
# ax.imshow(rgb, extent=[left, right, bottom, top], origin="upper")
# ax.set_xlabel("Longitude")
# ax.set_ylabel("Latitude")
# ax.set_title("RGB (2nd–98th percentile stretch)")

# def longitude_label(easting, _):
#     lon, _lat = transformer.transform(easting, centre_y)
#     return f"{abs(lon):.5f}°{'E' if lon >= 0 else 'W'}"

# def latitude_label(northing, _):
#     _lon, lat = transformer.transform(centre_x, northing)
#     return f"{abs(lat):.5f}°{'N' if lat >= 0 else 'S'}"

# ax.xaxis.set_major_formatter(FuncFormatter(longitude_label))
# ax.yaxis.set_major_formatter(FuncFormatter(latitude_label))
# ax.tick_params(axis="x", labelrotation=30)
# plt.tight_layout()
# plt.show()

## 6. Interactive band explorer

Choose a band to redraw its greyscale image and raw reflectance
histogram. Plot updates occur when the dropdown changes.


In [ ]:
# This is the display-resolution preview, not a second full-scene copy.
red650_val = ds["red650"].data.compute()
red650_val


In [ ]:
band_plot_info = {
    band["var"]: {
        "label": f"{band['name']} ({band['nm']} nm)",
        "dropdown_label": f"{band['name']} ({band['nm']} nm)",
        "colorscale": "Gray",
        "histogram_color": band["colour"],
        "value_label": "Surface reflectance",
    }
    for band in BAND_INFO
}

In [ ]:
fig_red650 = plot_raster_distribution(
    ds,
    variable="red650",
    metadata=band_plot_info,
    robust=False,
    bins=100,
    statistics=("min", "median", "max"),
    histogram_color="#c9342f",
    title="Red-650 Band and Histogram (display preview)",
)


# Finding no data values and masking them 

## For all bands (interactive)

In [ ]:
band_selector_full, band_output_full = interactive_raster_explorer(
    ds,
    variables=[band["var"] for band in BAND_INFO],
    metadata=band_plot_info,
    default="red668",
    dropdown_description="Band:",
    robust=False,
    bins=80,
    statistics=("mean", "p2", "p98"),
    show_colorbar=False,
    width=1200,
    height=570,
)


## 7. Build your own composite


In [ ]:
import numpy as np
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display, clear_output

# Use the original full dataset if ds was previously cropped.
source_ds = ds_full if "ds_full" in globals() else ds

composite_options = [
    (f"{band['name']} ({band['nm']} nm)", band["var"])
    for band in BAND_INFO
]

sel_r = widgets.Dropdown(options=composite_options, value="red668", description="Red:")
sel_g = widgets.Dropdown(options=composite_options, value="green560", description="Green:")
sel_b = widgets.Dropdown(options=composite_options, value="blue", description="Blue:")

stretch_low = widgets.FloatSlider(
    value=2, min=0, max=15, step=0.5,
    description="Lower clip (%):",
    continuous_update=False,
)
stretch_high = widgets.FloatSlider(
    value=98, min=85, max=100, step=0.5,
    description="Upper clip (%):",
    continuous_update=False,
)

custom_output = widgets.Output()
HISTOGRAM_BINS = 65536


def stretch_full_band_to_uint8(variable, low, high):
    """Calculate percentile limits and stretch one full band in chunks."""
    band_data = source_ds[variable].data
    height, width = source_ds.sizes["y"], source_ds.sizes["x"]
    histogram = np.zeros(HISTOGRAM_BINS, dtype=np.int64)

    if hasattr(band_data, "to_delayed"):
        delayed = band_data.to_delayed()
        y_chunks, x_chunks = band_data.chunks
        blocks = []
        y_offset = 0

        for i, chunk_height in enumerate(y_chunks):
            x_offset = 0
            for j, chunk_width in enumerate(x_chunks):
                blocks.append((
                    y_offset, x_offset, int(chunk_height), int(chunk_width),
                    delayed[i, j],
                ))
                x_offset += int(chunk_width)
            y_offset += int(chunk_height)
    else:
        blocks = [
            (
                row, col,
                min(256, height - row),
                min(512, width - col),
                band_data[row:row + 256, col:col + 512],
            )
            for row in range(0, height, 256)
            for col in range(0, width, 512)
        ]

    # First pass: estimate percentile thresholds from chunk histograms.
    for _, _, _, _, block in blocks:
        values = block.compute() if hasattr(block, "compute") else block
        values = np.asarray(values, dtype=np.float32)
        valid = np.isfinite(values) & (values > 0)

        if valid.any():
            histogram += np.histogram(
                np.clip(values[valid], 0.0, 1.0),
                bins=HISTOGRAM_BINS,
                range=(0.0, 1.0),
            )[0]

        del values, valid

    total = int(histogram.sum())
    if total == 0:
        return np.zeros((height, width), dtype=np.uint8)

    cumulative = np.cumsum(histogram, dtype=np.int64)
    ranks = [
        min(total, max(1, int(np.ceil(p / 100 * total))))
        for p in (low, high)
    ]
    bin_indices = [
        min(
            HISTOGRAM_BINS - 1,
            int(np.searchsorted(cumulative, rank)),
        )
        for rank in ranks
    ]
    lower, upper = [
        (index + 0.5) / HISTOGRAM_BINS for index in bin_indices
    ]
    scale = max(upper - lower, 1e-9)

    # Second pass: write stretched values into the full-size output channel.
    channel = np.zeros((height, width), dtype=np.uint8)

    for row, col, block_height, block_width, block in blocks:
        values = block.compute() if hasattr(block, "compute") else block
        values = np.asarray(values, dtype=np.float32)
        valid = np.isfinite(values) & (values > 0)

        stretched = np.clip((values - lower) / scale, 0.0, 1.0)
        stretched[~valid] = 0.0
        channel[row:row + block_height, col:col + block_width] = (
            stretched * 255
        ).astype(np.uint8)

        del values, valid, stretched

    return channel


def render_custom(change=None):
    with custom_output:
        clear_output(wait=True)

        low = stretch_low.value
        high = stretch_high.value
        height, width = source_ds.sizes["y"], source_ds.sizes["x"]

        rgb = np.empty((height, width, 3), dtype=np.uint8)

        # Reuse a channel if the same band is selected for multiple colours.
        channel_cache = {}
        for channel_index, variable in enumerate(
            (sel_r.value, sel_g.value, sel_b.value)
        ):
            if variable not in channel_cache:
                channel_cache[variable] = stretch_full_band_to_uint8(
                    variable, low, high
                )
            rgb[:, :, channel_index] = channel_cache[variable]

        context = projected_plot_context(source_ds)
        fig = go.Figure(
            make_projected_image(source_ds, rgb, context=context)
        )
        apply_projected_axes(
            fig, context, row=1, col=1, axis_number=1
        )
        fig.update_layout(
            title=(
                f"Full-resolution custom RGB — R={sel_r.value}, "
                f"G={sel_g.value}, B={sel_b.value}; "
                f"P{low:g}–P{high:g}"
            ),
            template="plotly_white",
            width=900,
            height=650,
            margin=dict(l=90, r=40, t=80, b=80),
        )
        fig.show(config={"displaylogo": False})

        del fig, rgb, channel_cache


for control in (sel_r, sel_g, sel_b, stretch_low, stretch_high):
    control.observe(render_custom, names="value")

display(widgets.VBox([
    widgets.HBox([sel_r, sel_g, sel_b]),
    widgets.HBox([stretch_low, stretch_high]),
    custom_output,
]))
render_custom()

## 8. Scene-wide spectral signature


In [ ]:
# Use the full-scene chunked means already calculated in `band_stats`.
# For a descriptive scene-wide spread, this cell computes standard deviation
# one band at a time and keeps only the resulting scalar in memory.
wavelengths = np.array([b["nm"] for b in BAND_INFO])
names = [b["name"] for b in BAND_INFO]
means = np.asarray(scene_means, dtype=float)
std_values = []
for band in BAND_INFO:
    std_task = dask_array.nanstd(ds[band["var"]].data)
    std_values.append(float(std_task.compute()))
stds = np.asarray(std_values, dtype=float)

fig_sig = go.Figure()
fig_sig.add_trace(go.Scatter(
    x=np.concatenate([wavelengths, wavelengths[::-1]]),
    y=np.concatenate([means + stds, (means - stds)[::-1]]),
    fill="toself", fillcolor="rgba(39,128,227,0.12)",
    line=dict(color="rgba(0,0,0,0)"), name="±1 standard deviation",
))
fig_sig.add_trace(go.Scatter(
    x=wavelengths, y=means, mode="lines+markers", text=names,
    line=dict(color="#2780e3", width=2.5), marker=dict(size=8),
    hovertemplate="<b>%{text}</b><br>λ=%{x} nm<br>Mean=%{y:.4f}<extra></extra>",
    name="Scene mean",
))
fig_sig.update_layout(
    title="Scene-Wide Spectral Signature — All 10 Bands",
    xaxis_title="Wavelength (nm)", yaxis_title="Mean surface reflectance",
    height=400,
)
fig_sig.show()


## 9. Drone image on a basemap


In [ ]:
import numpy as np
import folium
from rasterio.warp import transform_bounds

# Use the original full dataset, even if ds currently contains a crop.
scene_ds = ds_full if "ds_full" in globals() else ds


def stretch_preview_band(band, low=2, high=98):
    values = np.asarray(band, dtype=np.float32)
    valid = np.isfinite(values) & (values > 0)
    result = np.zeros(values.shape, dtype=np.uint8)

    if valid.any():
        vmin, vmax = np.percentile(values[valid], [low, high])
        if vmax > vmin:
            scaled = np.clip((values - vmin) / (vmax - vmin), 0, 1)
            result[valid] = (scaled[valid] * 255).astype(np.uint8)

    return result


max_preview_side = 500
map_step = max(
    1,
    int(np.ceil(max(scene_ds.sizes["y"], scene_ds.sizes["x"]) / max_preview_side)),
)

rgb_bands = ["red668", "green560", "blue"]
map_preview = scene_ds.isel(
    y=slice(0, None, map_step),
    x=slice(0, None, map_step),
)[rgb_bands].compute()

rgb_preview = np.dstack([
    stretch_preview_band(map_preview[name].data)
    for name in rgb_bands
])

x = np.asarray(scene_ds["x"].values)
y = np.asarray(scene_ds["y"].values)
dx = abs(float(np.median(np.diff(x))))
dy = abs(float(np.median(np.diff(y))))

left = float(x.min() - dx / 2)
right = float(x.max() + dx / 2)
bottom = float(y.min() - dy / 2)
top = float(y.max() + dy / 2)

west, south, east, north = transform_bounds(
    scene_ds.attrs["crs"],
    "EPSG:4326",
    left, bottom, right, top,
    densify_pts=21,
)

map_rgb = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=17,
    max_zoom=23,
)

# folium.TileLayer("openstreetmap", name="OpenStreetMap").add_to(map_rgb)
folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Esri World Imagery",
    name="Esri Satellite",
).add_to(map_rgb)

folium.raster_layers.ImageOverlay(
    image=rgb_preview,
    bounds=[[south, west], [north, east]],
    opacity=0.88,
    name="Full-scene RGB preview",
    interactive=True,
).add_to(map_rgb)

folium.LayerControl(collapsed=False).add_to(map_rgb)
map_rgb

In [ ]:
# This cell remains as a short reminder of where the interactive map appears.
display(Markdown("The interactive map is displayed in the previous cell."))


## Activities

1. Use the band explorer from Blue through NIR-842. Which band makes
   vegetation brightest, and does this match the spectral signature?
2. Recreate CIR with Red=NIR-842, Green=Red-668, Blue=Green-531.
3. Toggle OpenStreetMap and Esri Satellite and identify features that
   align with the drone composite.

Percentile stretching improves display contrast by preventing a few
extreme pixels from compressing the useful tonal range.
